# TT-17 — Random Forest Regressor 
## Dự đoán giá vé máy bay để tư vấn khách "nên mua bây giờ hay chờ"

Notebook này là **bản trình bày từng bước** của pipeline chính trong `src/train.py`.
Toàn bộ logic tái sử dụng (không viết lại) từ module `src/train.py` để đảm bảo
notebook và script sản xuất luôn đồng nhất kết quả.


In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from train import (
    load_data, run_eda, build_preprocessor, rmse,
    run_baselines, train_random_forest, tune_random_forest,
    run_cross_validation, rmse_vs_n_estimators,
    run_permutation_importance, run_pdp_days_left,
    run_prediction_interval, run_extrapolation_experiment,
    run_shuffled_target_sanity_check, try_xgboost_comparison,
    NUMERIC_COLS, CATEGORICAL_COLS, TARGET_COL, STRATIFY_COL, RANDOM_STATE,
)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

%matplotlib inline

DATA_PATH = Path("../data/Clean_Dataset.csv")
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(exist_ok=True)


## 1. Nạp dữ liệu + kiểm tra chất lượng

Bỏ cột index thừa và cột `flight`. `load_data()` giờ CŨNG kiểm tra null, dòng
trùng lặp, và giá trị bất thường (giá vé/duration ≤ 0, days_left < 0) — raise
lỗi cứng nếu dữ liệu có vấn đề nghiêm trọng, thay vì train mù trên dữ liệu lỗi.


In [ ]:
df = load_data(DATA_PATH)
df.head()

In [ ]:
df.describe(include='all').T

## 2. EDA — khám phá dữ liệu

- Giá theo `days_left`: kỳ vọng gần như phẳng từ 50→20 ngày, tăng vọt trong 7 ngày cuối.
- Boxplot theo `class`: Business đắt hơn Economy rất nhiều (biến mạnh nhất).
- Boxplot theo `airline`: chênh lệch giữa các hãng.


In [ ]:
run_eda(df, REPORTS_DIR)

from IPython.display import Image, display
display(Image(filename=str(REPORTS_DIR / "gia_theo_days_left.png")))
display(Image(filename=str(REPORTS_DIR / "boxplot_gia_theo_class.png")))
display(Image(filename=str(REPORTS_DIR / "boxplot_gia_theo_airline.png")))


## 3. Chia train/test CÓ PHÂN TẦNG + xây pipeline tiền xử lý

`class` là biến chi phối mạnh nhất tới giá — phân tầng (`stratify=class`) đảm
bảo tỉ lệ Economy/Business giống nhau ở cả train và test, tránh một tập bị
lệch ngẫu nhiên. Cây/rừng ngẫu nhiên **không cần scale** biến số.


In [ ]:
X = df[NUMERIC_COLS + CATEGORICAL_COLS]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=X[STRATIFY_COL]
)
print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

preprocessor = build_preprocessor()


## 4. Baseline — DummyRegressor, Linear Regression, Decision Tree đơn (TT-16)

Random Forest phải đánh bại cả 3 baseline này thì mới đáng dùng.


In [ ]:
baseline_results = run_baselines(X_train, X_test, y_train, y_test, preprocessor)
pd.DataFrame(baseline_results).T


## 5. ⭐ Tinh chỉnh siêu tham số (RandomizedSearchCV)

Bản v1 chỉ dùng tham số mặc định "đoán bừa". Bước này tìm kiếm có hệ thống
bằng k-fold cross-validation trên một tập con (để tiết kiệm thời gian), rồi
model cuối được refit trên TOÀN BỘ train với cấu hình tốt nhất.


In [ ]:
tune_result = tune_random_forest(
    X_train, y_train, build_preprocessor(),
    n_iter=20, cv_folds=3, search_sample_frac=0.35,
)
tune_result


## 6. Random Forest Regressor (cuối cùng, dùng tham số đã tinh chỉnh)

`oob_score_` cho ước lượng R² "miễn phí" từ các mẫu out-of-bag, gần như một
validation set không tốn thêm dữ liệu.


In [ ]:
best_params = tune_result["best_params"]
rf_pipe = train_random_forest(X_train, y_train, build_preprocessor(), **best_params)

pred_test = rf_pipe.predict(X_test)
from sklearn.metrics import mean_absolute_error, r2_score
print(f"Test RMSE = {rmse(y_test, pred_test):,.0f}")
print(f"Test MAE  = {mean_absolute_error(y_test, pred_test):,.0f}")
print(f"Test R2   = {r2_score(y_test, pred_test):.4f}")
print(f"OOB R2    = {rf_pipe.named_steps['model'].oob_score_:.4f}")


⚠️ **Kiểm tra rò rỉ dữ liệu:** nếu R² > 0.995 trên tập test, cần xem lại có cột
nào suy ra trực tiếp từ `price` hay không. Mục 8 dưới đây kiểm tra điều này
một cách ĐỊNH LƯỢNG (không chỉ dựa vào ngưỡng R²).


## 7. ⭐ Cross-validation cho model cuối

1 lần train/test split có thể "may rủi". 5-fold CV cho biết RMSE dao động bao nhiêu giữa các cách chia khác nhau.

In [ ]:
cv_pipe = Pipeline([
    ("prep", build_preprocessor()),
    ("model", RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE, **best_params)),
])
cv_result = run_cross_validation(cv_pipe, X, y, cv_folds=5)
cv_result


## 8. ⭐ Sanity check chống rò rỉ dữ liệu — shuffled-target test

Train lại trên target đã xáo trộn ngẫu nhiên. Nếu R² vẫn cao, CHẮC CHẮN có rò rỉ.

In [ ]:
sanity_result = run_shuffled_target_sanity_check(X_train, X_test, y_train, y_test, build_preprocessor())
sanity_result


## 9. RMSE theo số lượng cây — tìm điểm bão hoà

In [ ]:
curve = rmse_vs_n_estimators(X_train, X_test, y_train, y_test, build_preprocessor(), REPORTS_DIR)
display(Image(filename=str(REPORTS_DIR / "rmse_theo_so_cay.png")))
print("Điểm bão hoà ước lượng: n_estimators =", curve["diem_bao_hoa_uoc_luong"])


## 10. Permutation Importance — VÀ so sánh với Gini Importance

**Không dùng `feature_importances_` mặc định** vì nó thiên vị các biến phân loại có
nhiều mức giá trị. Bản v2 vẽ THÊM biểu đồ so sánh trực tiếp 2 phương pháp để
chứng minh điều này bằng số liệu thực tế, không chỉ nói suông.


In [ ]:
importance_result = run_permutation_importance(rf_pipe, X_test, y_test, REPORTS_DIR)
display(Image(filename=str(REPORTS_DIR / "permutation_importance.png")))
display(Image(filename=str(REPORTS_DIR / "so_sanh_permutation_vs_gini.png")))


## 11. Partial Dependence Plot cho `days_left`

Định lượng câu hỏi nghiệp vụ cốt lõi: **mua sớm 1 tuần tiết kiệm bao nhiêu tiền?**

In [ ]:
pdp_conclusion = run_pdp_days_left(rf_pipe, X_train, REPORTS_DIR)
display(Image(filename=str(REPORTS_DIR / "pdp_days_left.png")))
pdp_conclusion


## 12. Khoảng dự báo 10–90% từ phân phối dự đoán của các cây

In [ ]:
interval_result = run_prediction_interval(rf_pipe, X_test, y_test, REPORTS_DIR)
display(Image(filename=str(REPORTS_DIR / "khoang_du_bao.png")))
interval_result


## 13. Thí nghiệm ngoại suy — trên NHIỀU hồ sơ đại diện

Bản v1 chỉ thử trên 1 dòng dữ liệu đầu tiên. Bản v2 lấy trung bình trên 30 hồ
sơ khách ngẫu nhiên để kết luận đáng tin cậy hơn về mặt thống kê.


In [ ]:
extrap_result = run_extrapolation_experiment(rf_pipe, X_train, REPORTS_DIR, n_profiles=30)
display(Image(filename=str(REPORTS_DIR / "ngoai_suy_days_left.png")))
extrap_result


## 14. (Tuỳ chọn) So sánh với XGBoost Regressor (TT-19)

Chỉ chạy được nếu đã cài `xgboost` (`pip install xgboost`). Có early stopping.

In [ ]:
xgb_result = try_xgboost_comparison(X_train, X_test, y_train, y_test, build_preprocessor())
xgb_result

## 15. Lưu model

In [ ]:
import joblib
from pathlib import Path

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)
joblib.dump(rf_pipe, MODELS_DIR / "rf_reg.joblib")
print(f"Đã lưu model tại {MODELS_DIR / 'rf_reg.joblib'}")


## 16. Kết luận

Điền các con số thực tế sau khi chạy trên `Clean_Dataset.csv` thật:

- **Siêu tham số tốt nhất**: ... (từ `tune_result["best_params"]`)
- **oob_score_ / Test R²**: ... (kiểm tra > 0.95 nhưng không rò rỉ)
- **5-fold CV RMSE**: ... ± ...
- **Sanity check rò rỉ dữ liệu**: R² với target xáo trộn = ... (kỳ vọng ~0)
- **Biến quan trọng nhất theo permutation importance**: ...
- **Tiết kiệm khi mua sớm hơn 1 tuần** (theo PDP): ...
- **Độ phủ thực tế của khoảng dự báo 10–90%**: ... (kỳ vọng gần 80%)
- **Kết luận ngoại suy**: mô hình có bị kẹp trần khi `days_left=100` không (trên 30 hồ sơ đại diện)?

Xem thêm `tests/test_train.py` để chạy 21 bài kiểm thử tự động xác nhận pipeline
hoạt động đúng trên dữ liệu giả lập, không cần chờ `Clean_Dataset.csv` thật.
